In [1]:
import cv2
import numpy as np
from rtmlib import Body

COCO_CONNECTIONS = [
    (0, 1), (0, 2), (1, 3), (2, 4),           # голова
    (5, 7), (7, 9), (6, 8), (8, 10),           # руки
    (5, 6), (5, 11), (6, 12), (11, 12),        # торс
    (11, 13), (13, 15), (12, 14), (14, 16),    # ноги
]

# Пороги уверенности для отрисовки каждой точки.
# Ноги в воде часто размыты — низкая уверенность не должна двигать
# трекер, даже если точку ещё можно слегка показать.
_DEFAULT_THR = np.full(17, 0.35)
_DEFAULT_THR[7:11]  = 0.15   # локти и запястья
_DEFAULT_THR[11:13] = 0.18   # бёдра
_DEFAULT_THR[13:15] = 0.25   # колени
_DEFAULT_THR[15:17] = 0.30   # лодыжки

# Более строгие пороги для обновления фильтра по времени.
# Один плохой кадр с ударом ногой не сдвинет скелет ног.
_UPDATE_THR = _DEFAULT_THR.copy()
_UPDATE_THR[11:13] = 0.15
_UPDATE_THR[13:15] = 0.22
_UPDATE_THR[15:17] = 0.28

# Максимальный скачок точки за кадр в долях размера тела (торса).
# При кроле запястье двигается быстро, но главная ошибка — когда весь скелет
# «схлопывается» к центру, если рука прошла над головой:
# RTMPose уверенно ставит конечности у центра кадра.
# Лимиты для каждой точки отдельно: схлопывание отсекается,
# нормальное плавание проходит.
_MAX_JUMP_BY_JOINT = np.full(17, 1.10, dtype=np.float64)
_MAX_JUMP_BY_JOINT[0:5]   = 0.45   # голова и лицо — медленно
_MAX_JUMP_BY_JOINT[5:7]   = 0.45   # плечи — привязаны к торсу
_MAX_JUMP_BY_JOINT[7:9]   = 1.00   # локти
_MAX_JUMP_BY_JOINT[9:11]  = 1.40   # запястья — быстрее всего при выходе руки
_MAX_JUMP_BY_JOINT[11:13] = 0.45   # бёдра — привязаны к торсу
_MAX_JUMP_BY_JOINT[13:15] = 1.10   # колени
_MAX_JUMP_BY_JOINT[15:17] = 1.30   # лодыжки — удар ногой

# RTMPose-L body7, 384x288 — точнее YOLO-pose и MediaPipe на
# необычных позах (плавание, гимнастика и т.д.). Детектор — YOLOX-M.
_RTMPOSE_L_URL = (
    "https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/onnx_sdk/"
    "rtmpose-l_simcc-body7_pt-body7_420e-384x288-3f5a1437_20230504.zip"
)
_YOLOX_M_URL = (
    "https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/onnx_sdk/"
    "yolox_m_8xb8-300e_humanart-c2c7a14a.zip"
)

_model = Body(
    det=_YOLOX_M_URL,
    det_input_size=(640, 640),
    pose=_RTMPOSE_L_URL,
    pose_input_size=(288, 384),
    backend="onnxruntime",
    device="cpu",
)


def _preprocess(frame: np.ndarray) -> np.ndarray:
    """Улучшает контраст кадра и убирает размытие от движения."""
    lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    l = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8)).apply(l)
    enhanced = cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2BGR)
    blur = cv2.GaussianBlur(enhanced, (0, 0), sigmaX=2)
    return cv2.addWeighted(enhanced, 1.5, blur, -0.5, 0)


class _OneEuroFilter:
    """Фильтр One-Euro — http://cristal.univ-lille.fr/~casiez/1euro/

    Сглаживает шумные сигналы в реальном времени. Два параметра:
      * min_cutoff (Гц):  меньше = сильнее сглаживание, когда сигнал стабилен.
                         Попробуйте 0.5–2.0. Меньше — меньше дрожания в покое.
      * beta:            насколько быстро растёт частота среза при движении.
                         Больше — быстрее реакция, меньше задержка.
                         Попробуйте 0.005–0.05.
    Скорость тоже фильтруется (d_cutoff), чтобы один шумный кадр не давал
    скачок, который испортит оценку положения.
    Работает поэлементно с массивами numpy.
    """

    def __init__(self, freq: float = 30.0, min_cutoff: float = 1.0,
                 beta: float = 0.02, d_cutoff: float = 1.0):
        self.freq = freq
        self.min_cutoff = min_cutoff
        self.beta = beta
        self.d_cutoff = d_cutoff
        self.x_prev: np.ndarray | None = None
        self.dx_prev: np.ndarray | None = None

    def _alpha(self, cutoff):
        tau = 1.0 / (2 * np.pi * cutoff)
        te = 1.0 / self.freq
        return 1.0 / (1.0 + tau / te)

    def __call__(self, x: np.ndarray, update_mask: np.ndarray | None = None) -> np.ndarray:
        """x: сигнал формы (N,) или (N, D). update_mask: массив bool (N,) —
        обновлять только где True; где False — оставить прошлое значение."""
        if self.x_prev is None:
            self.x_prev = x.copy()
            self.dx_prev = np.zeros_like(x)
            return x.copy()

        dx = (x - self.x_prev) * self.freq
        a_d = self._alpha(self.d_cutoff)
        dx_hat = a_d * dx + (1 - a_d) * self.dx_prev

        cutoff = self.min_cutoff + self.beta * np.abs(dx_hat)
        a = self._alpha(cutoff)
        x_hat = a * x + (1 - a) * self.x_prev

        if update_mask is not None:
            if x_hat.ndim == 2 and update_mask.ndim == 1:
                update_mask_b = update_mask[:, None]
            else:
                update_mask_b = update_mask
            x_hat = np.where(update_mask_b, x_hat, self.x_prev)
            dx_hat = np.where(update_mask_b, dx_hat, self.dx_prev)

        self.x_prev = x_hat
        self.dx_prev = dx_hat
        return x_hat.copy()


def _estimate_body_center(kp_xy: np.ndarray, kp_conf: np.ndarray, thr: np.ndarray) -> tuple[np.ndarray | None, int]:
    """Центр торса по плечам и бёдрам; руки над головой его не сдвигают."""
    anchors = np.array([5, 6, 11, 12])
    ok = kp_conf[anchors] > thr[anchors]
    if ok.sum() < 2:
        return None, int(ok.sum())
    return kp_xy[anchors][ok].mean(axis=0), int(ok.sum())


def _estimate_body_scale(kp_xy: np.ndarray, kp_conf: np.ndarray, thr: np.ndarray) -> float:
    """Размер тела в пикселях; помогает отсечь нереальные скачки за один кадр."""
    pairs = [(5, 6), (11, 12), (5, 11), (6, 12)]
    lengths = []
    for a, b in pairs:
        if kp_conf[a] > thr[a] and kp_conf[b] > thr[b]:
            lengths.append(np.linalg.norm(kp_xy[a] - kp_xy[b]))
    if lengths:
        return float(np.median(lengths))
    return 0.0


def _kp_spread(kp_xy: np.ndarray, kp_conf: np.ndarray, thr: np.ndarray) -> float:
    """Диагональ рамки вокруг уверенно найденных точек.

    Когда RTMPose «схлопывает» позу (часто при руке над головой и странном
    кадре YOLOX), все точки сбиваются в одну кучу, и размах становится
    намного меньше тела. Это дополнительный признак, чтобы отклонить весь кадр,
    если проверка масштаба торса этого не заметила.
    """
    high = kp_conf > thr
    if int(high.sum()) < 4:
        return 0.0
    pts = kp_xy[high]
    return float(np.linalg.norm(pts.max(axis=0) - pts.min(axis=0)))


# «Кости» торса для детектора схлопывания. Каждая пара — индексы точек,
# расстояние между которыми должно быть примерно постоянным
# от кадра к кадру (с учётом поворота тела). Резкое укорочение любой из них —
# признак, что часть позы схлопнулась в одну точку.
# Медиана по костям и общий размах точек этого не ловят,
# если остальное тело на месте.
# (конец блока про схлопывание по костям)
_TORSO_BONES = ((5, 6), (11, 12), (5, 11), (6, 12))

# Сегменты конечностей для проверки формы. При кроле за один кадр
# запястье/лодыжка могут «переехать» в торс — длина нереальна,
# а торс выглядит нормально. EWMA длин помогает отбросить такие кадры.
# (конец блока про конечности)
_LIMB_BONES = (
    (5, 7), (7, 9), (6, 8), (8, 10),      # руки: плечо–локоть–запястье
    (11, 13), (13, 15), (12, 14), (14, 16) # ноги: бедро–колено–лодыжка
)


def _torso_bone_lengths(kp_xy: np.ndarray, kp_conf: np.ndarray, thr: np.ndarray) -> np.ndarray:
    """Длины костей торса (NaN, если конец ниже порога)."""
    out = np.full(len(_TORSO_BONES), np.nan, dtype=np.float64)
    for i, (a, b) in enumerate(_TORSO_BONES):
        if kp_conf[a] > thr[a] and kp_conf[b] > thr[b]:
            out[i] = float(np.linalg.norm(kp_xy[a] - kp_xy[b]))
    return out


def _limb_bone_lengths(kp_xy: np.ndarray, kp_conf: np.ndarray, thr: np.ndarray) -> np.ndarray:
    """Длины костей конечностей (NaN, если конец ниже порога)."""
    out = np.full(len(_LIMB_BONES), np.nan, dtype=np.float64)
    for i, (a, b) in enumerate(_LIMB_BONES):
        if kp_conf[a] > thr[a] and kp_conf[b] > thr[b]:
            out[i] = float(np.linalg.norm(kp_xy[a] - kp_xy[b]))
    return out


def _detect_swim_cap(
    frame: np.ndarray,
    prev_cap: np.ndarray | None = None,
    prev_r: float = 0.0,
    darkness_thr: int = 60,
    min_area: int = 800,
    max_area: int = 8000,
    max_aspect_ratio: float = 2.8,
    continuity_weight: float = 5.0,
    max_jump_px: float = 80.0,
    radius_change_factor: float = 0.5,
) -> tuple[np.ndarray | None, float]:
    """Ищет тёмную шапочку на кадре. Возвращает (центр_xy, радиус_в_пикселях).

    На видео сверху шапочка — самый стабильный ориентир для головы: вода светлая,
    кожа светлая, шапочка тёмная. Берём тёмные пиксели, ищем связные области
    подходящего размера и формы и выбираем лучший вариант. Если передан `prev_cap`,
    отбрасываем кандидатов, которые прыгнули дальше `max_jump_px` от прошлой
    шапочки или резко изменили радиус больше чем на `radius_change_factor` —
    так отсекаем блики, тени и частичное перекрытие.

    Возвращает (None, 0.0), если ничего не подошло.
    """
    if frame is None:
        return None, 0.0
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    _, mask = cv2.threshold(gray, darkness_thr, 255, cv2.THRESH_BINARY_INV)
    kern = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kern)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kern)

    n_labels, _, stats, cents = cv2.connectedComponentsWithStats(mask)
    best_xy: np.ndarray | None = None
    best_r = 0.0
    best_score = -np.inf
    for i in range(1, n_labels):  # 0 is background
        area = int(stats[i, cv2.CC_STAT_AREA])
        if area < min_area or area > max_area:
            continue
        w = int(stats[i, cv2.CC_STAT_WIDTH])
        h = int(stats[i, cv2.CC_STAT_HEIGHT])
        ar = max(w, h) / max(min(w, h), 1)
        if ar > max_aspect_ratio:
            continue
        cx, cy = float(cents[i, 0]), float(cents[i, 1])
        r = float(np.sqrt(area / np.pi))
        if prev_cap is not None:
            d = float(np.linalg.norm(np.array([cx, cy]) - prev_cap))
            if d > max_jump_px:
                continue
            if prev_r > 1.0 and abs(r - prev_r) > prev_r * radius_change_factor:
                continue
            score = area - continuity_weight * d
        else:
            score = float(area)
        if score > best_score:
            best_score = score
            best_xy = np.array([cx, cy], dtype=np.float64)
            best_r = r
    return best_xy, best_r


def _select_best_person(keypoints: np.ndarray, scores: np.ndarray,
                        prev_center: np.ndarray | None,
                        prev_scale: float | None) -> int:
    """Выбирает детекцию, которая лучше совпадает с уже отслеживаемым телом.

    Без прошлого кадра — с наибольшей средней уверенностью. С прошлым — смешивает
    уверенность и близость центра торса к прошлому центру (в долях размера тела),
    чтобы фантом на брызгах не перехватил трек у пловца.
    """
    n = int(len(keypoints))
    if n == 0:
        return 0
    if n == 1 or prev_center is None or prev_scale is None or prev_scale <= 1e-3:
        return int(np.argmax(scores.mean(axis=1)))

    best_idx = int(np.argmax(scores.mean(axis=1)))
    best_score = -np.inf
    for i in range(n):
        c, _ = _estimate_body_center(keypoints[i], scores[i], _UPDATE_THR)
        mean_s = float(scores[i].mean())
        if c is None:
            score = mean_s - 1.0
        else:
            dist = float(np.linalg.norm(c - prev_center)) / prev_scale
            score = mean_s - 0.3 * dist
        if score > best_score:
            best_score = score
            best_idx = i
    return best_idx


def _infer_pose_with_flip(
    frame: np.ndarray,
    prev_center: np.ndarray | None,
    prev_scale: float | None,
) -> tuple[np.ndarray | None, np.ndarray | None]:
    """Запускает RTMPose на кадре и на его повороте на 180°; берёт лучший результат.

    На кроле сверху RTMPose-L часто «переворачивает» пловца: вытянутая рука
    похожа на ногу, и голова оказывается на шортах, а лодыжки — на брызгах от руки.
    На практике правильная ориентация совпадает с проходом с более высокой
    средней уверенностью. Стоимость ~×2, зато ориентация верная.
    Координаты с повёрнутого кадра возвращаются в исходную систему.

    Лучший человек в каждом проходе выбирается через `_select_best_person` —
    ближе к уже отслеживаемому телу, чтобы фантом на брызгах не украл трек.

    Возвращает (kp_xy, kp_conf) формы (17, 2) / (17,) для лучшего человека
    в пикселях исходного кадра, или (None, None), если никого не нашли.
    """
    h, w = frame.shape[:2]

    def _eval(kp_arr, sc_arr):
        if kp_arr is None or len(kp_arr) == 0:
            return None, None, -np.inf
        idx = _select_best_person(kp_arr, sc_arr, prev_center, prev_scale)
        return kp_arr[idx], sc_arr[idx], float(sc_arr[idx].mean())

    kp_n, sc_n = _model(frame)
    best_n_xy, best_n_sc, score_n = _eval(kp_n, sc_n)

    kp_f, sc_f = _model(cv2.flip(frame, -1))
    if kp_f is not None and len(kp_f) > 0:
        kp_f = kp_f.copy()
        kp_f[..., 0] = w - kp_f[..., 0]
        kp_f[..., 1] = h - kp_f[..., 1]
    best_f_xy, best_f_sc, score_f = _eval(kp_f, sc_f)

    if not np.isfinite(score_n) and not np.isfinite(score_f):
        return None, None
    if score_f > score_n:
        return best_f_xy, best_f_sc
    return best_n_xy, best_n_sc


class _KPSmoother:
    """Сглаживает каждую координату (x, y); при пропуске — последняя известная поза.

    Видимые точки проходят через One-Euro; невидимые остаются на месте,
    уверенность постепенно падает — скелет плавно исчезает, если точка
    долго не находится. Пороги обновления и лимит скачка важны для кроля:
    быстрые размытые ноги часто дают правдоподобные, но неверные лодыжки
    на один-два кадра.
    """

    def __init__(self, fps: float = 30.0, min_cutoff: float = 1.0, beta: float = 0.02,
                 update_thr: np.ndarray | float = _UPDATE_THR, conf_decay: float = 0.85,
                 max_jump_per_joint: np.ndarray = _MAX_JUMP_BY_JOINT,
                 max_body_jump_scale: float = 1.10,
                 scale_ratio_range: tuple[float, float] = (0.45, 2.20),
                 min_spread_ratio: float = 1.35,
                 min_bone_ratio: float = 0.45,
                 limb_ratio_range: tuple[float, float] = (0.55, 1.80),
                 strict_conf_for_long_jump: float = 0.65,
                 min_trusted_joints: int = 6,
                 velocity_momentum: float = 0.75,
                 prediction_decay: float = 0.90,
                 hold_conf_frames: int = 10,
                 hold_conf_decay: float = 0.97,
                 max_missed_frames: int = 18,
                 bone_ewma_alpha: float = 0.30,
                 limb_ewma_alpha: float = 0.22,
                 cap_max_distance_scale: float = 1.0,
                 cap_min_scale_from_radius: float = 4.0):
        self.filter = _OneEuroFilter(freq=fps, min_cutoff=min_cutoff, beta=beta)
        update_thr = np.asarray(update_thr, dtype=np.float64)
        if update_thr.ndim == 0:
            update_thr = np.full(17, float(update_thr))
        self.update_thr = update_thr
        self.conf_decay = conf_decay
        self.max_jump_per_joint = np.asarray(max_jump_per_joint, dtype=np.float64)
        self.max_body_jump_scale = max_body_jump_scale
        self.scale_ratio_range = scale_ratio_range
        self.min_spread_ratio = min_spread_ratio
        self.min_bone_ratio = min_bone_ratio
        self.limb_ratio_range = limb_ratio_range
        self.strict_conf_for_long_jump = strict_conf_for_long_jump
        self.min_trusted_joints = min_trusted_joints
        self.velocity_momentum = velocity_momentum
        self.prediction_decay = prediction_decay
        self.hold_conf_frames = hold_conf_frames
        self.hold_conf_decay = hold_conf_decay
        self.max_missed_frames = max_missed_frames
        self.bone_ewma_alpha = bone_ewma_alpha
        self.limb_ewma_alpha = limb_ewma_alpha
        self.cap_max_distance_scale = cap_max_distance_scale
        self.cap_min_scale_from_radius = cap_min_scale_from_radius
        self._conf: np.ndarray | None = None    # (17,)
        self._xy: np.ndarray | None = None      # (17, 2)
        self._vel: np.ndarray | None = None     # (17, 2) px/s
        self._center: np.ndarray | None = None  # (2,)
        self._scale: float | None = None
        # EWMA длин костей торса по хорошим кадрам — для детектора схлопывания.
        # Инициализация с первого удачного кадра.
        self._bone_ewma: np.ndarray | None = None  # (4,)
        # EWMA длин конечностей — ловит нереальную геометрию рук и ног.
        self._limb_ewma: np.ndarray | None = None  # (8,)
        # Сколько кадров подряд точку «держим» без нового измерения.
        # Сигнал «потеряна» — уверенность не падает сразу, пока держим позу.

        self._missed_frames: np.ndarray = np.zeros(17, dtype=np.int32)
        # Якорь по шапочке: тёмная шапочка — самый стабильный объект сверху.
        # Проверяем, куда модель поставила голову (и передаём `_cap_xy` как
        # подсказку на следующий кадр). `_pose_anchor_cap` — где была шапочка,
        # когда зафиксировали `_xy`. Смещение шапочки сдвигает удерживаемую
        # позу при серии отклонённых кадров — скелет следует за пловцом.



        self._cap_xy: np.ndarray | None = None
        self._cap_r: float = 0.0
        self._pose_anchor_cap: np.ndarray | None = None

    def _predict_step(self, translate_to_cap: np.ndarray | None = None
                      ) -> tuple[np.ndarray | None, np.ndarray | None]:
        """Продвигает на один кадр без доверия к новым детекциям.

        Если передан `translate_to_cap` и есть прошлая шапочка, сдвигаем весь
        скелет на смещение шапочки — удерживаемая поза следует за головой.
        Иначе предсказываем по скорости (обычно через пару кадров поза замирает).
        """
        if self._xy is None or self._conf is None:
            return None, None
        if self._vel is None:
            self._vel = np.zeros_like(self._xy)

        if translate_to_cap is not None and self._pose_anchor_cap is not None:
            delta = translate_to_cap - self._pose_anchor_cap
            self._xy = self._xy + delta
            if self._center is not None:
                self._center = self._center + delta
            self._pose_anchor_cap = translate_to_cap.copy()
            # Скорость не имеет смысла, когда тянем позу за шапочкой.
            self._vel *= self.prediction_decay
        else:
            self._xy = self._xy + self._vel / max(self.filter.freq, 1e-6)
            self._vel *= self.prediction_decay

        next_miss = np.minimum(
            self._missed_frames + 1, np.iinfo(self._missed_frames.dtype).max
        )
        grace = next_miss <= self.hold_conf_frames
        floor = np.minimum(0.98, self.update_thr + 0.03)
        self._conf = np.where(grace, np.maximum(self._conf, floor), self._conf * self.hold_conf_decay)
        self._missed_frames = next_miss.astype(np.int32)
        return self._xy.copy(), self._conf.copy()

    def predict_only(self, current_cap: np.ndarray | None = None
                     ) -> tuple[np.ndarray | None, np.ndarray | None]:
        """Запасной вариант, когда детектор не нашёл человека.

        Если шапочка найдена, сдвигаем удерживаемую позу (см. `_predict_step`).
        """
        return self._predict_step(translate_to_cap=current_cap)

    def update(self, kp_xy: np.ndarray, kp_conf: np.ndarray,
               cap_xy: np.ndarray | None = None,
               cap_r: float = 0.0) -> tuple[np.ndarray, np.ndarray]:
        kp_xy = kp_xy.astype(np.float64)
        kp_conf = kp_conf.astype(np.float64)
        update = kp_conf > self.update_thr  # (17,) bool
        center, n_anchors = _estimate_body_center(kp_xy, kp_conf, self.update_thr)
        scale = _estimate_body_scale(kp_xy, kp_conf, self.update_thr)
        spread = _kp_spread(kp_xy, kp_conf, self.update_thr)
        bones = _torso_bone_lengths(kp_xy, kp_conf, self.update_thr)
        limb_bones = _limb_bone_lengths(kp_xy, kp_conf, self.update_thr)

        # Если торс резко сдвинулся или изменился в размере — скорее всего плохой
        # кадр, когда рука прошла над головой. Держим прошлую позу на этом кадре.

        reject_pose = False
        if self._center is not None and self._scale is not None:
            if center is None or n_anchors < 2 or scale <= 1e-3:
                reject_pose = True
            else:
                center_jump = np.linalg.norm(center - self._center)
                scale_ratio = scale / max(self._scale, 1e-3)
                lo, hi = self.scale_ratio_range
                reject_pose = (
                    center_jump > self.max_body_jump_scale * self._scale
                    or scale_ratio < lo
                    or scale_ratio > hi
                )
            # Схлопывание всего скелета: все уверенные точки в крошечной области.
            # Часто при неверном кадре YOLOX, когда рука над головой.

            if not reject_pose and spread > 0 and spread < self.min_spread_ratio * self._scale:
                reject_pose = True
            # Схлопывание по кости: самая длинная видимая кость торса намного
            # короче недавней EWMA. Берём max отношений (не any), чтобы отличить
            # настоящее схлопывание (все кости короче) от поворота тела: плечи
            # и бёдра укорачиваются, диагонали плечо–бедро почти прежние.
            # Поворот при каждом вдохе на кроле — не схлопывание, иначе EWMA
            # не обновится и скелет застынет.


            if not reject_pose and self._bone_ewma is not None:
                with np.errstate(invalid="ignore", divide="ignore"):
                    ratios = bones / self._bone_ewma
                ok_b = (
                    np.isfinite(ratios)
                    & np.isfinite(self._bone_ewma)
                    & (self._bone_ewma > 1e-3)
                )
                if ok_b.any() and float(ratios[ok_b].max()) < self.min_bone_ratio:
                    reject_pose = True

        # Мало точек прошло фильтр — кадр ненадёжен для обновления трекера.

        trusted_joints = int(update.sum())
        if trusted_joints < self.min_trusted_joints:
            reject_pose = True

        # Долго держим позу — она устарела, пловец уже ушёл. Пропускаем новую
        # детекцию, чтобы лимит скачка (с обходом для «потерянных» точек)
        # заново привязал скелет, а не застыл навсегда.

        # Проверки по шапочке ниже снова отклоняют кадр, если шапочка
        # противоречит модели — ослабляем только *временные* отказы, не внешние проверки.)

        if reject_pose and int(self._missed_frames.min()) >= self.max_missed_frames:
            reject_pose = False

        # Якорь шапочки (последний, имеет приоритет над повторным приёмом
        # после max_missed_frames): шапочка — единственное внешнее доказательство
        # положения головы; одних временных фильтров мало — RTMPose иногда
        # пропускает «перевёрнутые» позы.
        # Режим «голова на шортах» — кадр за кадром.
        # Если шапочку уже видели, но на *этом* кадре её нет — ориентацию
        # не проверить; отклоняем и предсказываем.

        if cap_xy is None and self._pose_anchor_cap is not None:
            reject_pose = True

        if cap_xy is not None:
            head_idx = np.array([0, 1, 2, 3, 4])
            head_ok = kp_conf[head_idx] > self.update_thr[head_idx]
            if head_ok.any():
                head_xy = kp_xy[head_idx][head_ok].mean(axis=0)
                ref_scale = max(
                    scale,
                    self._scale if self._scale is not None else 0.0,
                    float(cap_r) * self.cap_min_scale_from_radius,
                )
                if (ref_scale > 1e-3 and
                        float(np.linalg.norm(head_xy - cap_xy))
                        > self.cap_max_distance_scale * ref_scale):
                    reject_pose = True

            # Ориентация: бёдра модели должны быть дальше от шапочки, чем голова.
            # При «перевёрнутом» сбое RTMPose кладёт «бёдра» на настоящую голову
            # (на шапочку), а «голову» — на бёдра/шорты; отношение
            # d(голова, шапочка) / d(бёдра, шапочка) меняется с << 1 на >> 1.


            if head_ok.any():
                hip_idx = np.array([11, 12])
                hip_ok = kp_conf[hip_idx] > self.update_thr[hip_idx]
                if hip_ok.any():
                    hip_xy = kp_xy[hip_idx][hip_ok].mean(axis=0)
                    d_head_cap = float(np.linalg.norm(head_xy - cap_xy))
                    d_hip_cap = float(np.linalg.norm(hip_xy - cap_xy))
                    if d_hip_cap < max(d_head_cap, float(cap_r)) * 1.2:
                        reject_pose = True

        if reject_pose:
            if self._xy is not None and self._conf is not None:
                # На плохих кадрах предсказываем и держим уверенность короткое
                # «льготное» окно — точки не исчезают сразу при перекрытии.
                # Если есть шапочка — тянем позу за ней.

                xy_p, conf_p = self._predict_step(translate_to_cap=cap_xy)
                return xy_p, conf_p
            # Нет прошлого состояния, а первое измерение отклонено —
            # не инициализируем сглаживатель плохой позой. Ждём первый
            # кадр, прошедший фильтры.

            return None, None

        if not reject_pose:
            # Временной фильтр по точкам в *долях тела*:
            # вычитаем сдвиг центра тела между кадрами, потом смотрим скачок
            # каждой точки. Чистое перемещение пловца (и отставание камеры)
            # (пловец движется по кадру при неподвижной камере, или
            # не считается — только движение *относительно тела*. Без этого
            # жёсткий лимит на голову/плечи/бёдра отвергает все кадры при
            # движении, уверенность падает, точки исчезают с видео.



            # Точки, удерживаемые дольше `max_missed_frames`, считаются
            # «потерянными» и обходят фильтр — скелет снова поймает точку
            # после длинной серии сбоев.
            # Принимаем большой скачок, если модель даёт сильную новую
            # детекцию (conf >= strict_conf_for_long_jump) и удерживаемая
            # уверенность уже сильно ниже новых данных — запястье/локоть
            # (удержание < 0.4 * новое). Это для быстрого выхода руки, когда
            # реально смещается > body_scale за кадр, а лимит иначе
            # заморозит точку до конца клипа (напр. кадр 32: conf 0.81,
            # удержание 0.12, скачок 167px при лимите 112px).


            if self._xy is not None and scale > 1e-3:
                if (self._center is not None and center is not None):
                    body_shift = center - self._center
                    rel_jump = np.linalg.norm(
                        kp_xy - (self._xy + body_shift), axis=1
                    )
                else:
                    rel_jump = np.linalg.norm(kp_xy - self._xy, axis=1)
                lost = self._missed_frames > self.max_missed_frames
                strong_new = kp_conf >= self.strict_conf_for_long_jump
                if self._conf is not None:
                    strong_new &= self._conf < (kp_conf * 0.4)
                update &= (
                    lost
                    | strong_new
                    | (rel_jump < self.max_jump_per_joint * scale)
                )

            # Контроль формы конечности: если длина сегмента внезапно
            # нереальна относительно недавней геометрии — доверяем только
            # при очень высокой уверенности. Блокирует однокадровые сбои
            # запястья/лодыжки при брызгах, не мешая нормальному движению.
            if self._limb_ewma is not None:
                lo, hi = self.limb_ratio_range
                bad_joint = np.zeros(17, dtype=bool)
                for idx, (a, b) in enumerate(_LIMB_BONES):
                    cur = limb_bones[idx]
                    ref = self._limb_ewma[idx]
                    if not np.isfinite(cur) or not np.isfinite(ref) or ref <= 1e-3:
                        continue
                    ratio = cur / ref
                    if ratio < lo or ratio > hi:
                        suspect = a if kp_conf[a] <= kp_conf[b] else b
                        bad_joint[suspect] = True
                update &= ~(bad_joint & (kp_conf < self.strict_conf_for_long_jump))

        prev_xy = self._xy.copy() if self._xy is not None else None
        smoothed_xy = self.filter(kp_xy, update_mask=update)

        if self._conf is None:
            self._conf = np.where(update, kp_conf, 0.0)
        else:
            new_conf = self._conf.copy()
            new_conf[update] = kp_conf[update]
            new_conf[~update] *= self.conf_decay
            self._conf = new_conf

        # Обновляем скорости точек только по принятым измерениям.
        if self._vel is None:
            self._vel = np.zeros_like(smoothed_xy)
        if prev_xy is not None:
            dt = 1.0 / max(self.filter.freq, 1e-6)
            raw_vel = (smoothed_xy - prev_xy) / dt
            m = self.velocity_momentum
            up2 = update[:, None]
            self._vel = np.where(up2, m * self._vel + (1 - m) * raw_vel, self._vel * self.prediction_decay)

        # Реальное измерение сбрасывает счётчик пропусков; иначе +1.

        self._missed_frames = np.where(update, 0, self._missed_frames + 1).astype(np.int32)

        self._xy = smoothed_xy.copy()
        if cap_xy is not None:
            # Запоминаем шапочку в момент фиксации позы. `_predict_step`
            # сравнивает смещение с этим якорем и тянет позу за пловцом
            # при серии отклонённых кадров.

            self._pose_anchor_cap = cap_xy.copy()
        if not reject_pose and center is not None and scale > 1e-3:
            self._center = center.copy()
            self._scale = scale
            # EWMA длин костей торса только по принятым кадрам.
            valid_b = np.isfinite(bones)
            if self._bone_ewma is None:
                self._bone_ewma = np.where(valid_b, bones, np.nan)
            else:
                a = self.bone_ewma_alpha
                prev = self._bone_ewma
                new = np.where(
                    valid_b & np.isfinite(prev),
                    a * np.nan_to_num(bones) + (1 - a) * np.nan_to_num(prev),
                    np.where(valid_b, bones, prev),
                )
                self._bone_ewma = new

            # EWMA длин конечностей только по принятым кадрам.
            valid_l = np.isfinite(limb_bones)
            if self._limb_ewma is None:
                self._limb_ewma = np.where(valid_l, limb_bones, np.nan)
            else:
                a_l = self.limb_ewma_alpha
                prev_l = self._limb_ewma
                new_l = np.where(
                    valid_l & np.isfinite(prev_l),
                    a_l * np.nan_to_num(limb_bones) + (1 - a_l) * np.nan_to_num(prev_l),
                    np.where(valid_l, limb_bones, prev_l),
                )
                self._limb_ewma = new_l
        return smoothed_xy, self._conf


def _draw_person(frame, kp_xy, vis):
    for a, b in COCO_CONNECTIONS:
        if vis[a] and vis[b]:
            cv2.line(
                frame,
                (int(kp_xy[a, 0]), int(kp_xy[a, 1])),
                (int(kp_xy[b, 0]), int(kp_xy[b, 1])),
                (0, 200, 255), 2, cv2.LINE_AA,
            )
    for i, (x, y) in enumerate(kp_xy):
        if vis[i]:
            if i >= 11:
                color, r = (0, 80, 255), 5     # ноги — красный/оранжевый
            elif i in (7, 8, 9, 10):
                color, r = (255, 120, 0), 4    # руки — синий
            else:
                color, r = (0, 255, 0), 4      # голова/торс — зелёный
            cv2.circle(frame, (int(x), int(y)), r, color, -1)


def annotate_video(
    video_path: str,
    output_path: str = "annotated.mp4",
    kp_conf_thr: np.ndarray = _DEFAULT_THR,
    update_conf_thr: np.ndarray = _UPDATE_THR,
    min_cutoff: float = 1.0,
    beta: float = 0.02,
    max_jump_per_joint: np.ndarray = _MAX_JUMP_BY_JOINT,
    max_body_jump: float = 1.10,
    limb_ratio_range: tuple[float, float] = (0.55, 1.80),
    strict_conf_for_long_jump: float = 0.65,
    min_trusted_joints: int = 6,
    velocity_momentum: float = 0.75,
    prediction_decay: float = 0.90,
    hold_conf_frames: int = 10,
    hold_conf_decay: float = 0.97,
    preprocess: bool = True,
    use_orientation_flip: bool = True,
    use_cap_anchor: bool = False,
    cap_max_distance_scale: float = 1.0,
) -> str:
    """Распознаёт точки тела на каждом кадре с RTMPose-L и сохраняет видео с разметкой.

    RTMPose-L (384x288, body7) точнее YOLO-pose и MediaPipe на необычных позах,
    например при плавании. YOLOX-M находит человека, затем RTMPose работает
    по самому крупному кадру.

    Сглаживание — фильтр One-Euro: убирает дрожание в покое и быстро реагирует
    на резкие движения. Два параметра:
      * min_cutoff (Гц) — меньше = сильнее сглаживание в покое (0.5..2)
      * beta            — больше = меньше задержка при быстрых движениях (0.005..0.05)

    Аргументы:
        video_path:         Путь к исходному видео.
        output_path:        Путь к выходному видео с разметкой.
        kp_conf_thr:        Массив (17,) порогов уверенности для отрисовки.
        update_conf_thr:    Массив (17,) более строгих порогов для обновления трекера.
        min_cutoff:         Минимальная частота среза One-Euro.
        beta:               Коэффициент отзывчивости One-Euro.
        max_jump_per_joint: Массив (17,) лимитов скачка точки за кадр
                            в долях размера тела. Отсекает «схлопывание» конечностей,
                            когда рука проходит над головой и RTMPose ошибается.
        max_body_jump:      Максимальный скачок торса за кадр в долях размера тела.
        limb_ratio_range:   Допустимый диапазон длины конечности относительно EWMA.
                            Уже диапазон — жёстче отсеивает одиночные сбои.
        strict_conf_for_long_jump:
                            Выбросы по конечности принимаются только при уверенности
                            выше этого значения (0..1).
        min_trusted_joints: Минимум точек, прошедших фильтр, чтобы обновить трекер.
        velocity_momentum:  Инерция оценки скорости точки (0..1).
        prediction_decay:   Затухание скорости при отклонении позы.
        hold_conf_frames:   Сколько кадров держать точки видимыми после пропуска.
        hold_conf_decay:    Затухание уверенности после «льготного» окна.
        preprocess:         CLAHE + резкость перед распознаванием.
        use_orientation_flip:
                            Запуск RTMPose на кадре и на его повороте на 180°,
                            выбор лучшего результата. На кроле сверху чаще даёт
                            правильную ориентацию тела и живой скелет вместо
                            «перевёрнутой» позы. Включено по умолчанию — самое
                            заметное улучшение на этих записях.
        use_cap_anchor:     Ищет тёмную шапочку и отклоняет позы, где голова
                            слишком далеко от шапочки (или бёдра ближе к шапочке,
                            чем голова). По умолчанию выключено: при брызгах
                            шапочку теряют, и скелет «застывает». Полезно, если
                            шапочка стабильно видна.
        cap_max_distance_scale:
                            Максимальное расстояние от центра шапочки до головы
                            в долях размера тела. Меньше — строже.

    Возвращает:
        output_path
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Не удалось открыть видео: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS) or 20.0
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(output_path, fourcc, fps, (w, h))
    if not writer.isOpened():
        cap.release()
        raise RuntimeError(f"Не удалось создать видеофайл: {output_path}")

    smoother = _KPSmoother(
        fps=fps,
        min_cutoff=min_cutoff,
        beta=beta,
        update_thr=update_conf_thr,
        max_jump_per_joint=max_jump_per_joint,
        max_body_jump_scale=max_body_jump,
        limb_ratio_range=limb_ratio_range,
        strict_conf_for_long_jump=strict_conf_for_long_jump,
        min_trusted_joints=min_trusted_joints,
        velocity_momentum=velocity_momentum,
        prediction_decay=prediction_decay,
        hold_conf_frames=hold_conf_frames,
        hold_conf_decay=hold_conf_decay,
        cap_max_distance_scale=cap_max_distance_scale,
    )

    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        infer_frame = _preprocess(frame) if preprocess else frame

        # Шапочку ищем на *сыром* кадре — CLAHE при предобработке
        # осветляет тёмное и стирает контраст шапочки и кожи.
        # Прошлая шапочка не даёт случайной тёмной пятну перехватить якорь;
        # `prev_r` отсекает резкое изменение радиуса.

        if use_cap_anchor:
            cap_xy, cap_r = _detect_swim_cap(
                frame, prev_cap=smoother._cap_xy, prev_r=smoother._cap_r
            )
            if cap_xy is not None:
                smoother._cap_xy = cap_xy
                smoother._cap_r = cap_r
        else:
            cap_xy, cap_r = None, 0.0

        if use_orientation_flip:
            # Два прохода (обычный и повёрнутый): выбираем лучший
            # повёрнутый на 180°) дал более уверенную детекцию.
            # Восстанавливает правильную голову/ноги на кроле сверху,
            # где RTMPose иначе кладёт ноги на вытянутую руку.
            best_xy, best_sc = _infer_pose_with_flip(
                infer_frame, smoother._center, smoother._scale
            )
        else:
            keypoints, scores = _model(infer_frame)
            if keypoints is not None and len(keypoints) > 0:
                # Берём детекцию ближе к уже отслеживаемому телу —
                # фантом на брызгах не перехватит трек.
                idx = _select_best_person(
                    keypoints, scores, smoother._center, smoother._scale
                )
                best_xy, best_sc = keypoints[idx], scores[idx]
            else:
                best_xy = best_sc = None

        if best_xy is not None:
            kp_xy_s, kp_conf_s = smoother.update(
                best_xy, best_sc, cap_xy=cap_xy, cap_r=cap_r
            )
        else:
            # Человек не найден: держим трек предсказанием,
            # сдвигаем за шапочкой, если она есть.
            kp_xy_s, kp_conf_s = smoother.predict_only(current_cap=cap_xy)

        if kp_xy_s is not None and kp_conf_s is not None:
            vis = kp_conf_s > kp_conf_thr
            if vis.sum() >= 3:
                _draw_person(frame, kp_xy_s, vis)

        # Отладка: бледное жёлтое кольцо вокруг шапочки —
        # видно, когда якорь найден или нет.
        if cap_xy is not None and cap_r > 0:
            cv2.circle(frame, (int(cap_xy[0]), int(cap_xy[1])),
                       int(max(cap_r, 12)), (0, 255, 255), 2, cv2.LINE_AA)

        writer.write(frame)
        frame_idx += 1

    cap.release()
    writer.release()
    print(f"Готово — записано {frame_idx} кадров в {output_path}")
    return output_path


load /Users/alex/.cache/rtmlib/hub/checkpoints/yolox_m_8xb8-300e_humanart-c2c7a14a.onnx with onnxruntime backend
load /Users/alex/.cache/rtmlib/hub/checkpoints/rtmpose-l_simcc-body7_pt-body7_420e-384x288-3f5a1437_20230504.onnx with onnxruntime backend


In [3]:
annotate_video(
    video_path="../swimbot_data/Кроль сверху.mp4",
    output_path="annotated.mp4",
    limb_ratio_range=(0.68, 1.45),
    strict_conf_for_long_jump=0.75,
    min_trusted_joints=7,
    beta=0.012,
    min_cutoff=1.3,
    prediction_decay=0.90,
    hold_conf_frames=14,
    hold_conf_decay=0.985,
)

Done — 65 frames written to annotated.mp4


'annotated.mp4'